In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ==========================================================
# TREDENCE CASE STUDY - HIGHER ACCURACY SELF PRUNING NETWORK
# ==========================================================

import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==========================================================
# CONFIG
# ==========================================================

SEED = 42
BATCH_SIZE = 128
EPOCHS = 50
LR = 3e-4
LAMBDAS = [0.002, 0.004, 0.006]

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================================
# REPRODUCIBILITY
# ==========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ==========================================================
# DEVICE
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

# ==========================================================
# DATA
# ==========================================================

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_set = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_set = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_set,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ==========================================================
# PRUNABLE LINEAR
# ==========================================================

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(out_features, in_features)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

        self.gate_scores = nn.Parameter(
            torch.randn(out_features, in_features) - 1.5
        )

        nn.init.kaiming_uniform_(self.weight, a=0.01)

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        pruned_w = self.weight * gates
        return F.linear(x, pruned_w, self.bias)

    def gates(self):
        return torch.sigmoid(self.gate_scores).detach()

# ==========================================================
# MODEL
# ==========================================================

class SelfPruningNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(3072, 1024)
        self.bn1 = nn.BatchNorm1d(1024)

        self.fc2 = PrunableLinear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)

        self.fc3 = PrunableLinear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)

        self.fc4 = PrunableLinear(256, 10)

        self.dropout = nn.Dropout(0.2)

    def forward(self, x):

        x = x.view(x.size(0), -1)

        x = self.dropout(F.relu(self.bn1(self.fc1(x))))
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))
        x = self.dropout(F.relu(self.bn3(self.fc3(x))))
        x = self.fc4(x)

        return x

    def prunable_layers(self):
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self):

        total = torch.tensor(0.0, device=device)

        for layer in self.prunable_layers():
            total = total + torch.sigmoid(layer.gate_scores).sum()

        return total

    def compute_sparsity(self, threshold=0.05):

        vals = []

        for layer in self.prunable_layers():
            vals.append(layer.gates().flatten())

        vals = torch.cat(vals)

        return (vals < threshold).float().mean().item() * 100.0

    def all_gates(self):

        vals = []

        for layer in self.prunable_layers():
            vals.append(layer.gates().cpu().numpy().flatten())

        return np.concatenate(vals)

# ==========================================================
# EVALUATION
# ==========================================================

@torch.no_grad()
def evaluate(model):

    model.eval()

    correct = 0
    total = 0

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / total

# ==========================================================
# TRAINING
# ==========================================================

def run_experiment(lam):

    print("\n" + "="*60)
    print("Lambda =", lam)
    print("="*60)

    model = SelfPruningNet().to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=torch.cuda.is_available()
    )

    for epoch in range(1, EPOCHS + 1):

        model.train()
        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                outputs = model(images)

                ce_loss = F.cross_entropy(outputs, labels)
                sp_loss = model.sparsity_loss()

                loss = ce_loss + lam * sp_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        if epoch == 1 or epoch % 5 == 0:

            acc = evaluate(model)
            sp = model.compute_sparsity()

            print(
                "Epoch {}/{} | Loss {:.4f} | Acc {:.2f}% | Sparsity {:.1f}%".format(
                    epoch,
                    EPOCHS,
                    running_loss / len(train_loader),
                    acc,
                    sp
                )
            )

    final_acc = evaluate(model)
    final_sp = model.compute_sparsity()
    gates = model.all_gates()

    return {
        "lambda": lam,
        "accuracy": final_acc,
        "sparsity": final_sp,
        "gates": gates
    }

# ==========================================================
# RUN ALL
# ==========================================================

results = []

for lam in LAMBDAS:
    results.append(run_experiment(lam))

# ==========================================================
# RESULTS
# ==========================================================

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

rows = []

for r in results:
    rows.append([
        r["lambda"],
        round(r["accuracy"], 2),
        round(r["sparsity"], 2)
    ])

df = pd.DataFrame(
    rows,
    columns=["Lambda", "Accuracy", "Sparsity"]
)

print(df)

df.to_csv(
    os.path.join(SAVE_DIR, "summary.csv"),
    index=False
)

# ==========================================================
# PLOT
# ==========================================================

best = max(results, key=lambda x: x["accuracy"])

g = best["gates"]

plt.figure(figsize=(10,5))
plt.hist(g, bins=100, density=True)
plt.axvline(0.05, linestyle="--")
plt.title(
    "Gate Distribution | Lambda={} | Acc={:.2f}% | Sparsity={:.1f}%".format(
        best["lambda"],
        best["accuracy"],
        best["sparsity"]
    )
)
plt.xlabel("Gate Value")
plt.ylabel("Density")
plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "gate_distribution.png"),
    dpi=150
)

plt.show()

# ==========================================================
# JSON SAVE
# ==========================================================

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(rows, f)

print("\nSaved:")
print("summary.csv")
print("gate_distribution.png")
print("metrics.json")

print("\nDone.")